In [ ]:
# ── ЯЧЕЙКА 1 ──────────────────────────────────────────────────────────────
# ⏱ ~1 мин | 📌 Базовое подключение к YandexGPT

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv("/workspace/.env")

_api_key   = os.getenv("YANDEX_API_KEY")
_folder_id = os.getenv("YANDEX_FOLDER_ID")

llm = ChatOpenAI(
    base_url="https://llm.api.cloud.yandex.net/v1",
    api_key=_api_key,
    model=f"gpt://{_folder_id}/yandexgpt/latest",
    default_headers={"x-folder-id": _folder_id},
    temperature=0.1,
    max_tokens=512,
)

response = llm.invoke("Привет! Ты локальная модель?")
print(response.content)
print(f"Использовано токенов: {response.usage_metadata}")


In [ ]:
# ── ЯЧЕЙКА 2 ──────────────────────────────────────────────────────────────
# ⏱ ~2 мин | 📌 Первая LCEL цепочка: промпт → LLM → парсер

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты корпоративный ассистент службы поддержки. Отвечай чётко и по делу."),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"question": "Как сбросить пароль от корпоративной почты?"})
print(result)


In [ ]:
# ── ЯЧЕЙКА 3a ──────────────────────────────────────────────────────────────
# ⏱ ~2 мин | 📌 Streaming — токены приходят по одному в реальном времени
# ⚠️ Ключевой момент для UX: TTFT < 0.5 сек вместо ожидания 10-30 сек

print("Streaming (токены появляются по мере генерации):")
print("─" * 50)
for chunk in chain.stream({"question": "Объясни разницу между SQL и NoSQL за 3 предложения."}):
    print(chunk, end="", flush=True)
print("\n" + "─" * 50)


In [ ]:
# ── ЯЧЕЙКА 3b ──────────────────────────────────────────────────────────────
# ⏱ ~2 мин | 📌 Продвинутый LCEL конвейер: генерация + сайд-эффекты (запись файла)

import os
from typing import Iterator
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

def write_to_disk_stream(stream: Iterator[str]) -> Iterator[str]:
    """
    Правильный перехватчик (Middleware) для стриминга.
    Принимает поток чанков, пишет каждый на диск и прокидывает (yield) дальше,
    не блокируя работу LCEL-цепочки и не заставляя ждать конца генерации.
    """
    for chunk in stream:
        with open("/workspace/data/postmortem.md", "a", encoding="utf-8") as f:
            f.write(chunk)
        yield chunk  # Отдаем токен дальше по конвейеру (в консоль)

# Промпт: задаем строгую структуру ответа для системного инженера
prompt = ChatPromptTemplate.from_template("""
Ты — Senior SRE инженер. Составь шаблон Post-Mortem отчета для следующего инцидента.
Инцидент: {incident}

Требования:
- Строгий Markdown-формат.
- Обязательные секции: "Влияние", "Хронология", "Корневая причина", "Action Items".
- Без лишних приветствий, сразу к делу.
""")

llm = ChatOllama(model="qwen3:8b", base_url="http://localhost:11434", reasoning=False)

# Подготовка: создаем директорию и очищаем файл перед запуском
os.makedirs("/workspace/data", exist_ok=True)
with open("/workspace/data/postmortem.md", "w", encoding="utf-8") as f:
    f.write("<!-- АВТОГЕНЕРАЦИЯ POST-MORTEM -->\n\n")

# ── LCEL Pipeline: 5 компонентов конвейера ─────────────────────────────
# 1. RunnablePassthrough — берет входящую строку и оборачивает её в словарь
# 2. Prompt              — вставляет инцидент в шаблон
# 3. LLM                 — генерирует ответ
# 4. StrOutputParser     — срезает служебные теги моделей, возвращает чистый текст
# 5. write_to_disk_stream— пишет токены в файл, не прерывая стрим

postmortem_chain = (
    {"incident": RunnablePassthrough()}     # Шаг 1: str -> dict
    | prompt                                # Шаг 2: dict -> PromptValue
    | llm                                   # Шаг 3: PromptValue -> AIMessage
    | StrOutputParser()                     # Шаг 4: AIMessage -> str (поток)
    | RunnableLambda(write_to_disk_stream)  # Шаг 5: str -> str (с записью на диск)
)

print("ГЕНЕРАЦИЯ POST-MORTEM ОТЧЕТА (параллельно пишется в /workspace/data/postmortem.md):\n")

# Запускаем стриминг. Передаем просто строку, RunnablePassthrough сам положит её в ключ "incident"
test_incident = "Упал production-кластер Kubernetes из-за истекшего TLS сертификата etcd"

for chunk in postmortem_chain.stream(test_incident):
    print(chunk, end="", flush=True)

print("\n\n✅ Отчет успешно сохранен на диск!")


In [ ]:
# ── ЯЧЕЙКА 3c ──────────────────────────────────────────────────────────────
# ⏱ ~3 мин | 📌 Продвинутый LCEL конвейер с глубокой телеметрией (astream_events)

import os
from typing import AsyncIterator
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# 1. Асинхронный перехватчик (Middleware) для записи на диск
async def write_to_disk_stream(stream: AsyncIterator[str]) -> AsyncIterator[str]:
    """
    Генератор перехватывает токены на лету, пишет на диск и отдает дальше.
    """
    async for chunk in stream:
        with open("/workspace/data/postmortem.md", "a", encoding="utf-8") as f:
            f.write(chunk)
        yield chunk

# 2. Строгий промпт
prompt = ChatPromptTemplate.from_template("""
Ты — Senior SRE инженер. Составь шаблон Post-Mortem отчета для следующего инцидента.
Инцидент: {incident}

Требования:
- Строгий Markdown-формат.
- Обязательные секции: "Влияние", "Хронология", "Корневая причина", "Action Items".
- Без лишних приветствий, сразу к делу.
""")

llm = ChatOllama(model="qwen3:8b", base_url="http://localhost:11434", reasoning=False)

# 3. Подготовка файловой системы
os.makedirs("/workspace/data", exist_ok=True)
with open("/workspace/data/postmortem.md", "w", encoding="utf-8") as f:
    f.write("<!-- АВТОГЕНЕРАЦИЯ POST-MORTEM -->\n\n")

# 4. Сборка пайплайна
postmortem_chain = (
    {"incident": RunnablePassthrough()}          
    | prompt                                     
    | llm                                        
    | StrOutputParser()                          
    | write_to_disk_stream
)

print("ГЕНЕРАЦИЯ POST-MORTEM ОТЧЕТА (Глубокая телеметрия через astream_events):\n")

test_incident = "Упал production-кластер Kubernetes из-за истекшего TLS сертификата etcd"

# 5. ДЕМОНСТРАЦИЯ: Асинхронный перехват событий графа
async for event in postmortem_chain.astream_events(
    test_incident,
    version="v2"
):
    kind = event["event"]
    name = event.get("name", "")

    # Событие 1: Старт всей цепочки
    if kind == "on_chain_start" and name == "RunnableSequence":
        print("[⚙️ Инициализация пайплайна и подготовка промпта...]")
        
    # Событие 2: Модель начала "думать"
    elif kind == "on_chat_model_start":
        print("[▶ LLM начала генерацию...]\n")
        print("─" * 60)

    # Событие 3: Прилетел новый токен
    elif kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

    # Событие 4: Завершение работы нейросети
    elif kind == "on_chat_model_end":
        print("\n" + "─" * 60)
        print("[⏹ LLM успешно завершила генерацию]")

print("\n[✓ Файл /workspace/data/postmortem.md сохранен и содержит полный лог инцидента!]")


In [ ]:
# ── ЯЧЕЙКА 4a ──────────────────────────────────────────────────────────────
# ⏱ ~3 мин | 📌 Structured Output — гарантированный JSON
# ⚠️ .with_structured_output() — современный API (2025+)
# ⚠️ Модель физически НЕ МОЖЕТ вернуть невалидный JSON

from pydantic import BaseModel, Field
from typing import Literal

class SupportTicket(BaseModel):
    """Тикет поддержки — модель возвращает ИМЕННО ЭТУ структуру"""
    category: Literal["hardware", "software", "network", "access", "other"]
    priority: Literal["low", "medium", "high", "critical"]
    summary: str = Field(description="Краткое описание, 1 предложение")
    requires_callback: bool = Field(description="Нужен ли обратный звонок")

# Биндим Pydantic-схему к LLM
structured_llm = llm.with_structured_output(SupportTicket)

ticket = structured_llm.invoke(
    "У меня не работает VPN с утра. Завтра важная встреча с клиентом, нужно срочно. "
    "Телефон для связи оставил у секретаря."
)

print(f"Категория:      {ticket.category}")
print(f"Приоритет:      {ticket.priority}")
print(f"Описание:       {ticket.summary}")
print(f"Нужен звонок:   {ticket.requires_callback}")
print(f"\nJSON:\n{ticket.model_dump_json(indent=2)}")


In [ ]:
# ── ЯЧЕЙКА 4b ──────────────────────────────────────────────────────────────
# ⏱ ~3 мин | 📌 Structured Output + Семантическая маршрутизация (Router)
# Извлекаем строгий JSON из текста, показываем его и на лету маршрутизируем.

from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnableBranch, RunnableLambda
import datetime

# 1. Строгая схема данных (Контракт)
class SupportTicket(BaseModel):
    """Тикет поддержки — модель возвращает ИМЕННО ЭТУ структуру"""
    category: Literal["hardware", "software", "network", "access", "other"]
    priority: Literal["low", "medium", "high", "critical"]
    summary: str = Field(description="Краткое описание проблемы, 1 предложение")
    requires_callback: bool = Field(description="Нужен ли обратный звонок (если есть номер)")

# Оборачиваем LLM: теперь она гарантированно возвращает объект SupportTicket
structured_llm = llm.with_structured_output(SupportTicket)

# 2. ПРОКСИ-ПЕРЕХВАТЧИК ДЛЯ ДЕБАГА
def debug_and_pass(ticket: SupportTicket) -> SupportTicket:
    print(f"📦 [DEBUG] Извлечённый JSON:\n{ticket.model_dump_json(indent=2)}\n")
    return ticket

# 3. Специализированные узлы-обработчики (Handlers)
def handle_critical_network(ticket: SupportTicket) -> str:
    return f"🚨 [PAGERDUTY] СЕТЕВОЙ ИНЦИДЕНТ! Звоним дежурному NOC-инженеру.\n   Детали: {ticket.summary}"

def handle_access(ticket: SupportTicket) -> str:
    return f"🔑 [IAM AUTO-APPROVE] Запрос прав доступа. Отправлено на ревью тимлиду.\n   Описание: {ticket.summary}"

def handle_general(ticket: SupportTicket) -> str:
    return f"📝 [JIRA BACKLOG] Создан стандартный тикет (Приоритет: {ticket.priority.upper()}).\n   Категория: {ticket.category}"

# 4. Семантический маршрутизатор (RunnableBranch)
router = RunnableBranch(
    (lambda ticket: ticket.category == "network" and ticket.priority in ["high", "critical"],
     RunnableLambda(handle_critical_network)),
    (lambda ticket: ticket.category == "access",
     RunnableLambda(handle_access)),
    RunnableLambda(handle_general)
)

# 5. Супер-пайплайн с дебаг-узлом посередине
ticket_pipeline = structured_llm | RunnableLambda(debug_and_pass) | router

print("=== СЕМАНТИЧЕСКАЯ МАРШРУТИЗАЦИЯ ИНЦИДЕНТОВ ===\n")

# Тест 1: Завуалированная проблема с сетью
q1 = "У меня не работает VPN с утра. Завтра важная встреча с клиентом, нужно срочно. Звоните на мобильный."
print("◯ ЗАПРОС 1:", q1)
print("🔘 РЕЗУЛЬТАТ:\n" + ticket_pipeline.invoke(q1))
print("-" * 60)

# Тест 2: Запрос доступов
q2 = "Добавьте меня в группу разработчиков GitLab для проекта frontend-app."
print("◯ ЗАПРОС 2:", q2)
print("🔘 РЕЗУЛЬТАТ:\n" + ticket_pipeline.invoke(q2))
print("-" * 60)

# Тест 3: Бытовая проблема
q3 = "У меня мышка заедает, колесико не крутится. Замените при случае, не горит."
print("◯ ЗАПРОС 3:", q3)
print("🔘 РЕЗУЛЬТАТ:\n" + ticket_pipeline.invoke(q3))
print("-" * 60)

# ─── СВЯЗКА С ЯЧЕЙКОЙ 6c ─────────────────────────────────────────────────────
# Structured Output → файл на диске → read_ticket_file в агенте
#
# ЯЧЕЙКА 6c ожидает файл /workspace/ticket_TK2025001847.txt
# Здесь мы берём реальный запрос про FreeIPA/EMP042,
# прогоняем через structured_llm для классификации,
# и сохраняем на диск в формате который read_ticket_file прочитает.

print("\n" + "="*60)
print("🔗 СВЯЗКА С ЯЧЕЙКОЙ 6c")
print("   Structured Output → файл на диске → агент")
print("="*60)

# Исходный запрос — именно тот сценарий который агент будет обрабатывать
raw_query_for_agent = (
    "Необходимо срочно установить FreeIPA client на хост "
    "server-prod-app-04.corp.local для нового сотрудника "
    "Петрова Александра (EMP042, Senior DevOps Engineer). "
    "ОС: Astra Linux 1.8. Сотрудник выходит на работу завтра. "
    "Менеджер Сидоров Иван, звоните на мобильный."
)

print(f"\n📥 Входящий запрос:\n   {raw_query_for_agent}\n")

# structured_llm классифицирует запрос → SupportTicket объект
# Это демонстрирует как Structured Output питает downstream-системы
ticket_obj: SupportTicket = structured_llm.invoke(raw_query_for_agent)

print(f"📦 Structured Output (то что вернула LLM):")
print(ticket_obj.model_dump_json(indent=2))

# Форматируем в человекочитаемый тикет — именно этот формат
# будет читать read_ticket_file в ЯЧЕЙКЕ 6c
ticket_text = f"""ТИКЕТ #TK-2025-001847
Дата создания: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Приоритет: {ticket_obj.priority.upper()}
Категория: {ticket_obj.category}
Статус: ОТКРЫТ
Обратный звонок: {'Да' if ticket_obj.requires_callback else 'Нет'}

═══════════════════════════════════════════════════════════════════════════════

ТЕМА: {ticket_obj.summary}

ДЕТАЛИ ЗАПРОСА:
──────────────────────────────────────────────────────────────────────────────
- Имя сотрудника: Петров Александр (EMP042)
- Должность: Senior DevOps Engineer
- Отдел: Инфраструктура и облачные технологии
- Дата начала работы: завтра
- ОС хоста: Astra Linux 1.8
- Хост для подключения: server-prod-app-04.corp.local
- IP адрес хоста: 10.2.15.48

ТРЕБУЕМЫЕ ДЕЙСТВИЯ:
──────────────────────────────────────────────────────────────────────────────
1. Установить клиентское ПО FreeIPA на хост server-prod-app-04
2. Настроить аутентификацию Kerberos для домена corp.astra
3. Добавить EMP042 в группы: devops-team, infrastructure-admins, sudo-users

Менеджер по развитию: Сидоров Иван
Контакт: i.sidorov@corp.local
═══════════════════════════════════════════════════════════════════════════════
"""

# Сохраняем на диск — именно по пути который ожидает read_ticket_file в 6c
ticket_path = "/workspace/ticket_TK2025001847.txt"
with open(ticket_path, "w", encoding="utf-8") as f:
    f.write(ticket_text)

print(f"\n💾 Тикет сохранён на диск: {ticket_path}")
print(f"📏 Размер файла: {len(ticket_text)} символов")
print(f"\n📋 Содержимое файла:")
print("─" * 60)
print(ticket_text)
print("─" * 60)
print("\n✅ ГОТОВО — ЯЧЕЙКА 6c можно запускать")
print("   read_ticket_file найдёт файл по пути:")
print(f"   {ticket_path}")


In [ ]:
# ── ЯЧЕЙКА 5 ──────────────────────────────────────────────────────────────
# ⏱ ~3 мин | 📌 Отказоустойчивость (Fallbacks) + Телеметрия + Стриминг
# Graceful Degradation: бесшовное переключение на резервную модель с отладкой событий.

import asyncio
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

# 1. Эмулируем "упавший" облачный/внешний API через асинхронную функцию.
# Это гарантирует, что мы не зависнем на уровне сетевых сокетов ОС,
# а упадем ровно через 5 секунд (наш искусственный таймаут).
async def simulate_broken_api(prompt_value):
    await asyncio.sleep(5)
    raise ConnectionError("Timeout 504: Cloud LLM Gateway is down!")

# Оборачиваем функцию и даем ей имя для красивых логов в телеметрии
broken_primary_llm = RunnableLambda(simulate_broken_api).with_config(run_name="Cloud_GPT_Primary")

# 2. Наша рабочая резервная модель (Backup).
llm = llm
# llm = ChatOllama(model="qwen3:8b", base_url="http://localhost:11434", reasoning=False)
backup_llm = llm.with_config(run_name="Local_Qwen3_Backup")

# 3. ДЕМОНСТРАЦИЯ WITH_FALLBACKS:
# Привязываем резерв. LangChain сам перехватит ошибку из Cloud_GPT 
# и прозрачно перенаправит промпт в Local_Qwen3.
resilient_llm = broken_primary_llm.with_fallbacks([backup_llm])

# 4. Собираем стандартную LCEL-цепочку
prompt = ChatPromptTemplate.from_template(
    "Ты Senior DevOps. Объясни коротко (1-2 предложения): что такое {concept} в Kubernetes?"
)

resilient_chain = prompt | resilient_llm | StrOutputParser()

print("=== Демонстрация Fallback Mechanism + Телеметрия ===")
print("🌐 Запрос отправлен в пайплайн...\n")

# 5. ЗАПУСК СО СТРИМИНГОМ И ПЕРЕХВАТОМ СОБЫТИЙ
async for event in resilient_chain.astream_events(
    {"concept": "Pod Disruption Budget (PDB)"},
    version="v2"
):
    kind = event["event"]
    name = event.get("name", "")
    
    # Событие: Началось обращение к основной "облачной" модели
    if kind == "on_chain_start" and name == "Cloud_GPT_Primary":
        print(f"[{name}] ⏳ Ожидание ответа от облака (таймаут 5 сек)...", flush=True)
        
    # Событие: Основная модель УПАЛА С ОШИБКОЙ
    elif kind == "on_chain_error" and name == "Cloud_GPT_Primary":
        err_msg = event["data"].get("error")
        print(f"[{name}] 💥 УПАЛ С ОШИБКОЙ: {err_msg}")
        print("[LangChain] 🔄 Автоматически переключаю на резервную модель...")
        
    # Событие: Резервная локальная модель начала работу
    elif kind == "on_chat_model_start" and name == "Local_Qwen3_Backup":
        print(f"\n[{name}] ▶ Начала генерацию ответа:\n")
        print("─" * 60)
        
    # Событие: Стриминг токенов от резервной модели
    elif kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)
        
    # Событие: Резервная модель закончила работу
    elif kind == "on_chat_model_end" and name == "Local_Qwen3_Backup":
        print("\n" + "─" * 60)
        print(f"[{name}] ⏹ Успешно завершила генерацию")

print("\n✅ Пайплайн отработал без остановки скрипта (Graceful Degradation)!")
